In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [25]:
from gensim.models import Word2Vec
import pandas as pd
import os
import numpy as np

In [61]:
os.chdir('/content/drive/My Drive/NLP-Letters-V2/notebooks/')
df = pd.read_csv('../data/df_sentences_with_gender.csv')

# Balance Classes

In [64]:
# Separate majority and minority classes
majority_class = df[df['label'] == 1]
minority_class = df[df['label'] == 0]

# Downsample majority class
majority_downsampled = majority_class.sample(n=len(minority_class), random_state=42)

# Combine minority class with downsampled majority class
balanced_df = pd.concat([majority_downsampled, minority_class])

# Shuffle the DataFrame
balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Train Word2Vec

In [68]:
sentences = balanced_df['sentences'].tolist()

In [69]:
sentences = [line.split() for line in sentences]

In [105]:
wv2 = Word2Vec(sentences, vector_size=100, epochs=20, window=5)

# Analyze Results

In [106]:
wv2.wv.most_similar("he")

[('she', 0.7897725701332092),
 ('first_name', 0.6744498610496521),
 ('identifier', 0.494180291891098),
 ('“identifier', 0.48855432868003845),
 ('and', 0.4835340976715088),
 ('him', 0.48350462317466736),
 ('dr.identifier', 0.47091835737228394),
 ('“he', 0.4228702485561371),
 ('mr.', 0.4137837290763855),
 ('who', 0.4091457724571228)]

In [107]:
wv2.wv.most_similar("she")

[('he', 0.7897725701332092),
 ('first_name', 0.6530506610870361),
 ('identifier', 0.5006121397018433),
 ('and', 0.47402605414390564),
 ('“identifier', 0.46764498949050903),
 ('ms.', 0.4641396701335907),
 ('her', 0.4594116806983948),
 ('dr.identifier', 0.45838305354118347),
 ('ding', 0.43950971961021423),
 ('who', 0.4050639569759369)]

## Projections

In [108]:
# Get vectors for gendered words
he_vec = wv2.wv['he']
she_vec = wv2.wv['she']

# Calculate the gender axis
gender_axis = he_vec - she_vec

# Function to calculate projection on the gender axis
def get_projection(word):
    word_vec = wv2.wv[word]
    projection = np.dot(word_vec, gender_axis) / np.linalg.norm(gender_axis)
    return projection

# Example words to project
words_to_project = ['leader', 'caring', 'considerate', 'friendly', 'liked', 'gentle', 'nurturing', 'intelligent', 'supportive', 'brilliant', 'empathy', 'competent', 'warm', 'confident', 'mature', 'professional', 'kind', 'helpful']

# Calculate projections
projections = {word: get_projection(word) for word in words_to_project}

In [109]:
projections

{'leader': -0.74925596,
 'caring': 0.28957656,
 'considerate': 0.37837762,
 'friendly': 0.55711967,
 'liked': 0.9357923,
 'gentle': -0.69752616,
 'nurturing': 0.08666764,
 'intelligent': -0.008408823,
 'supportive': 0.503826,
 'brilliant': 0.24225327,
 'empathy': -0.023141565,
 'competent': -0.12070333,
 'warm': -0.9642566,
 'confident': 0.10575376,
 'mature': 0.26900145,
 'professional': 0.6221294,
 'kind': 0.6506084,
 'helpful': 0.028566534}

## WEAT

In [80]:

# Define target and attribute words
target_words_male = ['he']
target_words_female = ['she']
attribute_words_competent = ['leader', 'brilliant', 'intelligent', 'ambitious', 'confident']
attribute_words_warmth = ['caring', 'supportive', 'nurturing', 'helpful', 'team-player']

# Function to calculate cosine similarity
def cosine_similarity(vec1, vec2):
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

# Function to calculate WEAT score
def calculate_weat(target_words, attribute_set1, attribute_set2):
    scores = []
    for target in target_words:
        target_vec = wv2.wv[target]
        s1 = np.mean([cosine_similarity(target_vec, wv2.wv[attr]) for attr in attribute_set1])
        s2 = np.mean([cosine_similarity(target_vec, wv2.wv[attr]) for attr in attribute_set2])
        scores.append(s1 - s2)
    return np.mean(scores)

# Calculate WEAT scores for male and female targets
weat_male = calculate_weat(target_words_male, attribute_words_competent, attribute_words_warmth)
weat_female = calculate_weat(target_words_female, attribute_words_competent, attribute_words_warmth)

print("WEAT score (Male):", weat_male)
print("WEAT score (Female):", weat_female)

WEAT score (Male): 0.048454076
WEAT score (Female): 0.027973533
